# Punto 3: TSP Óptimo con Python (Held-Karp + Fuerza Bruta)

Resolución del Problema del Viajero (TSP) utilizando **solo Python** para encontrar la solución **óptima**.

## Algoritmos implementados

1. **Fuerza Bruta** (`itertools.permutations`): Evalúa todas las permutaciones posibles. Complejidad O(n!).
2. **Held-Karp** (Programación Dinámica con bitmask): Complejidad O(n² · 2ⁿ). Mucho más eficiente que fuerza bruta.

Se prueban subconjuntos de 4, 6, 8, 10, 12, 14, 16, 18 y 20 ciudades colombianas.

## 1. Carga de datos

In [ ]:
import json
import time
from itertools import permutations

DATA_URL = "https://raw.githubusercontent.com/d0ubt/problema-del-viajero-bd2/main/shared/ciudades.json"

try:
    with open("../shared/ciudades.json", "r") as f:
        data = json.load(f)
except FileNotFoundError:
    import urllib.request
    with urllib.request.urlopen(DATA_URL) as resp:
        data = json.loads(resp.read().decode())

ciudades = data["ciudades"]
dist_raw = data["distancias_km"]
subsets = data["subsets"]

n_total = len(ciudades)
dist_full = [[0] * n_total for _ in range(n_total)]
for key, val in dist_raw.items():
    a, b = key.split("-")
    a, b = int(a), int(b)
    dist_full[a][b] = val
    dist_full[b][a] = val

nombres = [c["nombre"] for c in ciudades]

print(f"Ciudades cargadas: {n_total}")
for c in ciudades:
    print(f"  {c['id']:>2}: {c['nombre']}")

## 2. Definición de algoritmos

In [ ]:
def build_sub_matrix(dist_matrix, indices):
    n = len(indices)
    sub = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            sub[i][j] = dist_matrix[indices[i]][indices[j]]
    return sub


def held_karp(dist, n):
    INF = float("inf")
    full_mask = (1 << n) - 1
    dp = [[INF] * n for _ in range(1 << n)]
    parent = [[-1] * n for _ in range(1 << n)]
    dp[1][0] = 0

    for mask in range(1, 1 << n):
        if not (mask & 1):
            continue
        for u in range(n):
            if not (mask & (1 << u)):
                continue
            if dp[mask][u] == INF:
                continue
            for v in range(n):
                if mask & (1 << v):
                    continue
                new_mask = mask | (1 << v)
                new_cost = dp[mask][u] + dist[u][v]
                if new_cost < dp[new_mask][v]:
                    dp[new_mask][v] = new_cost
                    parent[new_mask][v] = u

    best_cost = INF
    last_city = -1
    for u in range(1, n):
        cost = dp[full_mask][u] + dist[u][0]
        if cost < best_cost:
            best_cost = cost
            last_city = u

    if best_cost == INF:
        return None, INF

    route = []
    mask = full_mask
    curr = last_city
    while curr != -1:
        route.append(curr)
        prev = parent[mask][curr]
        mask = mask ^ (1 << curr)
        curr = prev
    route.reverse()
    route.append(0)
    return route, best_cost


def brute_force(dist, n):
    if n == 1:
        return [0, 0], 0
    cities = list(range(1, n))
    best_cost = float("inf")
    best_perm = None
    for perm in permutations(cities):
        cost = dist[0][perm[0]]
        for i in range(len(perm) - 1):
            cost += dist[perm[i]][perm[i + 1]]
        cost += dist[perm[-1]][0]
        if cost < best_cost:
            best_cost = cost
            best_perm = perm
    route = [0] + list(best_perm) + [0]
    return route, best_cost


print("Algoritmos definidos: held_karp() y brute_force()")

## 3. Ejecución de benchmarks

In [ ]:
sizes = [4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26]
results = []

for size in sizes:
    key = str(size)
    if key not in subsets:
        continue
    indices = subsets[key]
    sub_dist = build_sub_matrix(dist_full, indices)
    sub_nombres = [nombres[i] for i in indices]

    print(f"\n{'='*80}")
    print(f"Subconjunto de {size} ciudades: {', '.join(sub_nombres)}")
    print(f"{'='*80}")

    bf_cost = None
    bf_time = None
    bf_route_names = None
    if size <= 10:
        t0 = time.perf_counter()
        bf_route, bf_cost = brute_force(sub_dist, size)
        t1 = time.perf_counter()
        bf_time = t1 - t0
        bf_route_names = [sub_nombres[c] for c in bf_route]
        print(f"  Fuerza Bruta: {bf_cost} km | {bf_time:.6f}s")
        print(f"  Ruta: {' → '.join(bf_route_names)}")
    else:
        print(f"  Fuerza Bruta: OMITIDA (n={size} demasiado grande)")

    hk_cost = None
    hk_time = None
    hk_route_names = None
    hk_feasible = True
    try:
        t0 = time.perf_counter()
        hk_route, hk_cost = held_karp(sub_dist, size)
        t1 = time.perf_counter()
        hk_time = t1 - t0
        if hk_route is not None:
            hk_route_names = [sub_nombres[c] for c in hk_route]
            print(f"  Held-Karp:    {hk_cost} km | {hk_time:.6f}s")
            print(f"  Ruta: {' → '.join(hk_route_names)}")
        else:
            hk_feasible = False
            print(f"  Held-Karp: Sin solución")
    except MemoryError:
        hk_feasible = False
        print(f"  Held-Karp: ERROR DE MEMORIA")

    match = None
    if bf_cost is not None and hk_cost is not None:
        match = bf_cost == hk_cost
        print(f"  Coinciden: {'SI' if match else 'NO'}")

    results.append({
        "size": size,
        "bf_cost": bf_cost,
        "bf_time": bf_time,
        "bf_route": bf_route_names,
        "hk_cost": hk_cost,
        "hk_time": hk_time,
        "hk_route": hk_route_names,
        "hk_feasible": hk_feasible,
        "match": match,
    })

## 4. Tabla resumen

In [ ]:
print(f"\n{'Ciudades':>10} | {'BF Dist (km)':>14} | {'BF Tiempo (s)':>14} | {'HK Dist (km)':>14} | {'HK Tiempo (s)':>14} | {'Coinciden':>10}")
print("-" * 90)
for r in results:
    bf_d = f"{r['bf_cost']}" if r["bf_cost"] is not None else "N/A"
    bf_t = f"{r['bf_time']:.6f}" if r["bf_time"] is not None else "N/A"
    hk_d = f"{r['hk_cost']}" if r["hk_cost"] is not None else "N/A"
    hk_t = f"{r['hk_time']:.6f}" if r["hk_time"] is not None else "N/A"
    m = "SI" if r["match"] == True else ("NO" if r["match"] == False else "-")
    print(f"{r['size']:>10} | {bf_d:>14} | {bf_t:>14} | {hk_d:>14} | {hk_t:>14} | {m:>10}")

## 5. Análisis de factibilidad

In [ ]:
print("\nAnálisis de factibilidad:")
print("-" * 50)
for r in results:
    status = "FACTIBLE" if r["hk_feasible"] else "NO FACTIBLE"
    time_str = f"{r['hk_time']:.6f}s" if r["hk_time"] is not None else "N/A"
    print(f"  n={r['size']:>2}: Held-Karp {status} ({time_str})")

infeasible = [r for r in results if not r["hk_feasible"]]
if infeasible:
    print(f"\nHeld-Karp se vuelve no factible a partir de n={infeasible[0]['size']} ciudades.")
else:
    print(f"\nHeld-Karp fue factible para todos los tamaños probados (hasta n={results[-1]['size']}).")

bf_expensive = [r for r in results if r["bf_time"] is not None and r["bf_time"] > 1.0]
if bf_expensive:
    print(f"Fuerza Bruta se vuelve costosa (>1s) a partir de n={bf_expensive[0]['size']} ciudades.")
else:
    feasible_bf = [r for r in results if r["bf_time"] is not None]
    if feasible_bf:
        print(f"Fuerza Bruta fue rápida para todos los tamaños probados (hasta n={feasible_bf[-1]['size']}).")